# 🎯 03 - Evaluación y Chat Interactivo (RotBot English Coach)

Este notebook evalúa el modelo ajustado frente a una batería de **10 casos de prueba representativos** de hispanohablantes e incluye un chat interactivo con el tutor.

### Criterios de Evaluación para Rotbot:
1. **Personalidad e Identidad:** ¿Llama al usuario 'boss'? ¿Mantiene un tono sarcástico, inteligente y amigable?
2. **Estilo:** ¿Cumple con la restricción de **cero emojis** y respuesta 100% en inglés?
3. **Eficacia Pedagógica:** ¿Corrige el error de raíz con humor y claridad?

In [ ]:
# 1. Inicialización y carga de entorno
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.path.abspath("")).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

from src.parser import DEFAULT_SYSTEM_PROMPT
print("✅ Módulos cargados.")
print(f"ID de Modelo Tuned en .env: {os.getenv('TUNED_MODEL_ID', 'No configurado (usando base model)')}")
print(f"\nSystem Prompt Activo:\n{DEFAULT_SYSTEM_PROMPT}")

## 2. Batería de 10 Casos de Prueba Automatizados
Pruebas de interferencia español-inglés extraídas del enfoque pedagógico de Rotbot.

In [ ]:
TEST_SUITE = [
    {
        "id": 1,
        "category": "Age Expression",
        "input": "Is it okay to say 'I have 25 years'?",
        "expected_pattern": "am 25 years old"
    },
    {
        "id": 2,
        "category": "Feelings / States",
        "input": "I want to say I'm hungry, is 'I have hunger' correct?",
        "expected_pattern": "hungry"
    },
    {
        "id": 3,
        "category": "Verb Redundancy (Agree)",
        "input": "I wrote 'I am agree with you', can you check it?",
        "expected_pattern": "I agree"
    },
    {
        "id": 4,
        "category": "3rd Person Singular",
        "input": "Is 'She don't like coffee' correct?",
        "expected_pattern": "doesn't"
    },
    {
        "id": 5,
        "category": "Adjective -ing vs -ed",
        "input": "'I am boring in this class', I mean I'm getting bored",
        "expected_pattern": "bored"
    },
    {
        "id": 6,
        "category": "Collocation (Make vs Do mistake)",
        "input": "Can you check 'He did the same error again'?",
        "expected_pattern": "made"
    },
    {
        "id": 7,
        "category": "False Friend (Actually)",
        "input": "Is 'Actually I'm working in a bank' using 'actually' correctly?",
        "expected_pattern": "currently"
    },
    {
        "id": 8,
        "category": "Preposition (Depend on)",
        "input": "Is 'It's depend on the weather' correct?",
        "expected_pattern": "depends on"
    },
    {
        "id": 9,
        "category": "Plural Nouns (People)",
        "input": "Is 'The people is very nice here' correct?",
        "expected_pattern": "people are"
    },
    {
        "id": 10,
        "category": "Uncountable Nouns (Advice)",
        "input": "Can you check 'I need advices for my interview'?",
        "expected_pattern": "advice"
    }
]

print(f"📋 Total de pruebas preparadas: {len(TEST_SUITE)}")

## 3. Función de Inferencia del Coach

In [ ]:
def query_rotbot(user_text: str, model_id: str = None) -> str:
    """
    Envía un mensaje a RotBot utilizando la API configurada (Gemini, OpenAI o fallback).
    """
    gemini_key = os.getenv("GEMINI_API_KEY")
    openai_key = os.getenv("OPENAI_API_KEY")
    
    # 1. Gemini
    if gemini_key and not gemini_key.startswith("your_"):
        try:
            import google.generativeai as genai
            genai.configure(api_key=gemini_key)
            target_model = model_id or os.getenv("TUNED_MODEL_ID") or "gemini-1.5-flash"
            model = genai.GenerativeModel(
                model_name=target_model,
                system_instruction=DEFAULT_SYSTEM_PROMPT
            )
            response = model.generate_content(user_text)
            return response.text
        except Exception as e:
            print(f"[Gemini Error]: {e}")
            
    # 2. OpenAI
    if openai_key and not openai_key.startswith("your_"):
        try:
            from openai import OpenAI
            client = OpenAI(api_key=openai_key)
            target_model = model_id or os.getenv("TUNED_MODEL_ID") or "gpt-4o-mini"
            response = client.chat.completions.create(
                model=target_model,
                messages=[
                    {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                    {"role": "user", "content": user_text}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"[OpenAI Error]: {e}")
            
    return f"[MODO DEMO]: Para '{user_text}', configura tu GEMINI_API_KEY o OPENAI_API_KEY en .env."

## 4. Ejecución del Benchmark

In [ ]:
print("🚀 Iniciando evaluación de la batería de pruebas...\n")
for test in TEST_SUITE:
    print(f"------------------------------------------------------------")
    print(f"🧪 Caso #{test['id']} [{test['category']}]")
    print(f"👤 Input: \"{test['input']}\"")
    print(f"🎯 Esperado: '{test['expected_pattern']}' + personalidad Rotbot ('boss')")
    response = query_rotbot(test['input'])
    print(f"🤖 RotBot:\n{response}\n")

## 5. Chat Interactivo en Vivo con RotBot

In [ ]:
def interactive_chat_session():
    print("=" * 60)
    print("💬 Sesión interactiva con RotBot English Coach")
    print("Escribe tu mensaje en inglés (o 'salir' / 'exit' para terminar).")
    print("=" * 60)
    
    while True:
        try:
            user_input = input("\n👤 Tú: ").strip()
            if user_input.lower() in ('salir', 'exit', 'quit', 'q'):
                print("\n👋 RotBot: Take care, boss. Go practice your English.")
                break
            if not user_input:
                continue
                
            reply = query_rotbot(user_input)
            print(f"\n🤖 RotBot:\n{reply}")
        except KeyboardInterrupt:
            print("\n👋 Sesión finalizada.")
            break

# Descomenta la siguiente línea para iniciar el chat en la consola interactiva:
# interactive_chat_session()